<div style="font-size:11px; line-height:1.4">
<b style="font-size:13px">YOLO11-nano 自動化訓練 Notebook (本機)</b><br>
依 <code>dataset/</code> 訓練一個 <b>nano (yolo11n)</b> YOLO11 偵測模型，於專案根目錄輸出 <b>model.pt</b>。<br>
資料集：4 類別 (bus, car, motorcycle, truck)，train 164 / valid 32 / test 13。<br>
含雜訊/模糊增強，並<b>特別強化 motorcycle 召回率</b>（過採樣 4x + imgsz 1280 + 高 cls 損失 + 強增強）。<br><br>
<b style="font-size:12px">前置 (Python 3.10 + .venv)</b><br>
首次在專案根目錄 PowerShell 執行：<br>
<code>py -3.10 -m venv .venv</code><br>
<code>.\.venv\Scripts\Activate.ps1</code><br>
<code>pip install ultralytics albumentations ipykernel</code><br>
之後在 VSCode / Jupyter 選 <code>.venv</code> kernel，由上而下執行所有 cell。<br>
<i>註：本機為 Intel UHD 630（CPU 訓練），imgsz 1280 會非常慢；建議改用 train_colab.ipynb 於 GPU 訓練。</i>
</div>

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">1. 安裝相依套件</b><br>ultralytics 會自動帶入相容的 PyTorch；albumentations 供雜訊/模糊增強。</div>

In [ ]:
%pip install -q ultralytics albumentations

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">2. 匯入套件與設定路徑</b></div>

In [ ]:
from pathlib import Path
import shutil
import yaml
import torch
from ultralytics import YOLO

PROJECT = Path.cwd()                 # 專案根目錄 (train.ipynb 所在位置)
DATASET = PROJECT / "dataset"
SRC_YAML = DATASET / "data.yaml"

assert SRC_YAML.exists(), f"找不到資料集設定檔: {SRC_YAML}"
print("專案根目錄:", PROJECT)
print("資料集設定:", SRC_YAML)

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">3. 產生修正版 data.yaml</b><br>原始 <code>data.yaml</code> 用相對路徑 (<code>../train/images</code>) 且無 <code>path:</code> 鍵，Ultralytics 易解析錯誤。這裡用絕對路徑重寫一份 <code>data_fixed.yaml</code>，不修改原檔。</div>

In [ ]:
cfg = yaml.safe_load(SRC_YAML.read_text(encoding="utf-8"))

fixed = {
    "path": str(DATASET).replace("\\", "/"),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": cfg["nc"],
    "names": cfg["names"],
}

FIXED_YAML = DATASET / "data_fixed.yaml"
FIXED_YAML.write_text(yaml.safe_dump(fixed, allow_unicode=True, sort_keys=False), encoding="utf-8")

print("已產生:", FIXED_YAML)
print(FIXED_YAML.read_text(encoding="utf-8"))

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">4. 強化 motorcycle：影像過採樣 4x</b><br>掃描 train 標註，含 motorcycle (class 2) 的影像在每個 epoch 重複取樣 4 次，提升其相對頻率以改善召回。產生 <code>train_oversample.txt</code> 並把 yaml 的 <code>train</code> 指向它（Ultralytics 支援以影像清單 .txt 當 train，重複行＝重複取樣）。調整 <code>OVERSAMPLE</code> 可改變倍率。</div>

In [ ]:
import random

MOTO = cfg["names"].index("motorcycle")   # = 2
OVERSAMPLE = 4                            # 含 motorcycle 的影像重複次數
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

img_dir = DATASET / "train" / "images"
lbl_dir = DATASET / "train" / "labels"

lines, n_moto = [], 0
for img in sorted(img_dir.iterdir()):
    if img.suffix.lower() not in IMG_EXT:
        continue
    lbl = lbl_dir / (img.stem + ".txt")
    has_moto = lbl.exists() and any(
        ln.split()[0] == str(MOTO) for ln in lbl.read_text().splitlines() if ln.strip()
    )
    reps = OVERSAMPLE if has_moto else 1
    lines += [str(img.resolve())] * reps
    n_moto += has_moto

random.seed(0)
random.shuffle(lines)
TRAIN_LIST = DATASET / "train_oversample.txt"
TRAIN_LIST.write_text("\n".join(lines), encoding="utf-8")

fixed["train"] = str(TRAIN_LIST.resolve())               # 改用過採樣清單
FIXED_YAML.write_text(yaml.safe_dump(fixed, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(f"含 motorcycle 影像: {n_moto}，過採樣後訓練樣本數: {len(lines)} (原 164)")

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">5. 裝置自動偵測</b><br>有 CUDA GPU 用 GPU，否則退回 CPU。</div>

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
if device == 0:
    print("使用 GPU:", torch.cuda.get_device_name(0))
else:
    print("使用 CPU (未偵測到 CUDA GPU，imgsz 1280 會非常慢；建議改用 train_colab.ipynb)")

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">6. 資料增強設定 (雜訊 / 模糊)</b><br>包裝 Ultralytics 的 Albumentations 注入自訂 transform（Blur、MedianBlur、GaussNoise、ISONoise）。這些屬非空間變換，只改影像、不動 bbox。調整下方 <code>p</code> 可改變套用機率。其餘內建增強仍保留。本 cell 必須在訓練前執行。</div>

In [ ]:
import albumentations as A
from ultralytics.data import augment as _aug

# 保存原始 __init__，避免重複執行本 cell 時把包裝後的版本當成原始版
_orig_init = getattr(_aug.Albumentations, "_orig_init", _aug.Albumentations.__init__)
_aug.Albumentations._orig_init = _orig_init

def _patched_init(self, p=1.0, transforms=None):   # 接受 Ultralytics 傳入的 transforms 並改用自訂清單
    custom = [
        A.Blur(blur_limit=7, p=0.10),         # 一般模糊
        A.MedianBlur(blur_limit=7, p=0.10),   # 中值模糊
        A.GaussNoise(p=0.20),                 # 高斯雜訊
        A.ISONoise(p=0.10),                   # 感光雜訊
    ]
    _aug.Albumentations._orig_init(self, p=p, transforms=custom)

_aug.Albumentations.__init__ = _patched_init
print("已注入雜訊/模糊增強，訓練 log 的 albumentations: 行會列出 Blur/MedianBlur/GaussNoise/ISONoise")

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">7. 載入 nano 模型並訓練 (強化設定)</b><br>首次會自動下載 <code>yolo11n.pt</code>。強化 motorcycle 召回：<code>imgsz=1280</code>(小物件)、<code>cls=1.0</code>(分類損失)、<code>scale/mosaic/mixup</code>(強增強)、<code>patience=30</code>(early stopping)。<br><i>CPU 上 1280 非常慢，可暫時把 imgsz 改回 640、epochs 改小做冒煙測試。</i></div>

In [ ]:
model = YOLO("yolo11n.pt")           # nano 尺寸

results = model.train(
    data=str(FIXED_YAML),
    epochs=150,
    patience=30,                     # early stopping
    imgsz=1280,                      # 高解析度，利於小物件 (motorcycle) 召回
    cls=1.0,                         # 提高分類損失權重 (預設 0.5)
    scale=0.9, mosaic=1.0, mixup=0.1,  # 加強尺度/拼接/混合增強
    device=device,
    project=str(PROJECT / "runs"),
    name="train",
    exist_ok=True,
)

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">8. 將 best.pt 複製為根目錄 model.pt</b></div>

In [ ]:
best = Path(results.save_dir) / "weights" / "best.pt"
target = PROJECT / "model.pt"

assert best.exists(), f"找不到訓練輸出的 best.pt: {best}"
shutil.copy(best, target)
print("已輸出模型權重:", target)

<div style="font-size:11px; line-height:1.4"><b style="font-size:13px">9. 逐類召回率評估 (重點看 motorcycle)</b><br>用低 <code>conf</code> 在 test 集評估並印出每類 P/R。部署時調低偵測 <code>conf</code> 門檻可進一步提升召回。</div>

In [ ]:
m = YOLO(str(PROJECT / "model.pt"))
metrics = m.val(data=str(FIXED_YAML), split="test", imgsz=1280, conf=0.001, device=device)

print(f"{'class':<12}{'P':>8}{'R':>8}{'mAP50':>8}")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{m.names[c]:<12}{metrics.box.p[i]:>8.3f}{metrics.box.r[i]:>8.3f}{metrics.box.ap50[i]:>8.3f}")
print(f"\nall mAP50-95: {metrics.box.map:.3f}")